# 04 — The improved model & honest uncertainty

This is the headline system (rubric: Improved model). We take the ~293-Elo no-engine baseline from
notebook 03 and add the full 77-feature vector (engine move-quality + clock + scramble), early
stopping, and — crucially — a **calibrated prediction interval** rather than a bare point estimate.

**What you will see:**
1. The improvement table: **293.4 → 239.1 → 237.2** (no-engine → +engine/clock → tuned).
2. **SHAP** and **feature-group ablations** — what the model actually keys on (move quality
   dominates: dropping the engine group costs +33.6 MAE).
3. **Conformalized quantile regression (CQR)** — how a raw 87.4%-covering interval becomes a
   **guaranteed 90% marginal** interval, and what Mondrian (band-conditional) CQR does on top.
4. The **aggregation curve** — the headline chart: one game gives ±~235 Elo; five games give ~200.

> **Reproducibility.** By default this notebook reads a precomputed snapshot of the evaluation
> (`reports/eval_artifacts.json`) plus the saved figures in `reports/figures/`, both produced by
> `python -m src.evaluate` on the 300k data. Headline numbers are cross-quoted from
> `reports/results.md`. Set `RUN_HEAVY = True` in the setup cell to retrain the full model live
> (~15–20 min) and refresh the figures — the pipeline is deterministic (fixed seed), so the numbers
> match. *Notebooks are for looking; `src/evaluate.py` is for doing.*

In [ ]:
# --- Setup ----------------------------------------------------------------------------------
import sys, json, math, re, warnings
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.config import load_config
from src.plotting import set_style, save_fig

warnings.filterwarnings("ignore")
cfg = load_config()
set_style()
PROC = Path(cfg["paths"]["processed"])
FIGROOT = Path(cfg["paths"]["figures"])          # src.evaluate writes the pipeline figures here
REPORTS = FIGROOT.parent
FIG = FIGROOT / "notebook_04_model"              # this notebook's own deck figures
BANDS = cfg["rating_bands"]
FEAT = PROC / "features_300k_v2.parquet"
GAMES = PROC / "games_clean_300k.parquet"

def show(path):
    display(Image(filename=str(path)))

RUN_HEAVY = False   # flip to True to retrain the full model live (~15-20 min) and refresh figures
if RUN_HEAVY:
    from src.evaluate import run_evaluation
    _ = run_evaluation(cfg, str(FEAT), str(GAMES), tune=False, make_figures=True, results_md=None)
    print("Refreshed reports/figures/ from a live run.")

# Structured evaluation snapshot (produced by src.evaluate at the commit below) powers the tables/plots.
A = json.loads((REPORTS / "eval_artifacts.json").read_text(encoding="utf-8"))
print("artifact commit:", A["source_commit"], "| n_test:", f"{A['n_test']:,}", "| features:", A["n_features"])

# Minimal results.md reader so the headline numbers are literally quoted from the experiment log.
def latest_md_section(substr):
    blocks = re.split(r"(?m)^## ", (REPORTS / "results.md").read_text(encoding="utf-8"))[1:]
    hits = [b for b in blocks if substr in b.splitlines()[0]]
    return hits[-1] if hits else None

## 1. The improvement table

The full feature set (engine + clock + scramble) is what turns a mediocre surface model into a useful
one. The tuned row is Optuna's best on the same grouped splits; its hyperparameters are logged with
the run.

In [ ]:
# baseline/full from the artifact; the tuned row is quoted from reports/results.md.
tuned_block = latest_md_section("improved model (tuned)")
tuned_mae = float(re.search(r"\+ tuned \| ([\d.]+)", tuned_block).group(1))
tuned_rmse = float(re.search(r"\+ tuned \| [\d.]+ \| ([\d.]+)", tuned_block).group(1))
best_params = re.search(r"best_params: `(.+)`", tuned_block).group(1)

imp = pd.DataFrame([
    ("no-engine baseline", round(A["baseline_mae"], 1), float("nan")),
    ("+ engine/clock",      round(A["full_mae"], 1),     round(A["full_rmse"], 1)),
    ("+ Optuna tuned",      round(tuned_mae, 1),          round(tuned_rmse, 1)),
], columns=["stage", "MAE", "RMSE"]).set_index("stage")
display(imp)

fig, ax = plt.subplots(figsize=(7.5, 4.2))
bars = ax.bar(imp.index, imp["MAE"], color=["#9aa0a6", "#2c6fbb", "#3aa66f"])
for b, v in zip(bars, imp["MAE"]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}", ha="center", va="bottom", fontweight="bold")
ax.set(ylabel="MAE (Elo)", title="Improvement table (lower = better)", ylim=(0, 320))
ax.tick_params(axis="x", rotation=8)
show(save_fig(fig, "improved_improvement_table", FIG))
print("tuned best_params:", best_params)

**Reading it.** Engine + clock features cut MAE by **54 Elo (~18%)** over the no-engine baseline — the
single biggest jump in the project — and Optuna tuning adds a further ~2 Elo. The tuning gain is
small because the untuned defaults were already sensible; the features, not the hyperparameters, do
the work. We report the untuned full model as the honest headline and keep the tuned number as the
best achievable.

In [ ]:
# Fuller point metrics (a picture beyond MAE) + the per-player (all-games-averaged) number.
p, pp = A["point"], A["per_player"]
metrics = pd.DataFrame({
    "value": [round(p["mae"], 1), round(p["rmse"], 1), round(p["median_ae"], 0), round(p["r2"], 3),
              round(p["spearman"], 3), f"{p['within_100']*100:.0f}%", f"{p['within_200']*100:.0f}%",
              f"{p['bias']:+.1f}", round(pp["mae"], 1), f"{pp['n_players']:,}"],
}, index=["MAE", "RMSE", "median AE", "R^2", "Spearman rho", "within 100 Elo", "within 200 Elo",
          "bias", "per-player MAE (all games)", "n players"])
display(metrics)

**Reading it.** The full model explains R² ≈ 0.55 of rating variance with a rank correlation of ρ ≈
0.73, is within 200 Elo half the time on a *single* game, and is essentially unbiased globally (−3.4
Elo). Averaging *all* of a player's games drops the error to **218** — a first hint that aggregation,
not a cleverer single-game model, is the real lever (§4). Median AE (202) sits below MAE (239),
confirming a right-skewed error driven by the hard tail cases notebook 05 dissects.

## 2. What the model keys on — SHAP & ablations

Two complementary views: SHAP attributes each prediction to features (global importance + direction),
while a **group ablation** drops a whole feature family, retrains, and measures the MAE it was worth.

In [ ]:
# SHAP summary from the live pipeline (reports/figures/shap_summary.png, refreshed by RUN_HEAVY).
show(FIGROOT / "shap_summary.png")

In [ ]:
# Feature-group ablations: MAE increase when each group is dropped and the model retrained.
ab = pd.DataFrame(A["ablations"]).sort_values("mae_increase", ascending=False)
fig, ax = plt.subplots(figsize=(7.8, 4))
bars = ax.barh(ab["group"], ab["mae_increase"], color="#2c6fbb")
for b, v, n in zip(bars, ab["mae_increase"], ab["n_dropped"]):
    ax.text(v, b.get_y() + b.get_height() / 2, f"  +{v:.1f}  ({n} feats)", va="center", fontsize=9)
ax.invert_yaxis()
ax.set(xlabel="MAE increase when the group is dropped (Elo)",
       title="Feature-group ablation: what each family is worth")
show(save_fig(fig, "improved_ablation", FIG))
display(ab[["group", "n_dropped", "mae_without", "mae_increase"]].round(1).reset_index(drop=True))

**Reading it.** Move quality is the workhorse: SHAP puts centipawn-loss and accuracy features
(overall and post-book) at the top, and dropping the whole **engine** group costs **+33.6 MAE** — more
than the other three groups combined. **Clock** (+9.1), **opening** (`eco`, +8.2), and **style**
(+7.9) each add a real, smaller increment. The ordering matches the EDA in notebook 01: how well you
play separates ratings most, with time management and opening choice as secondary signals.

## 3. Honest uncertainty: conformalized quantile regression (CQR)

A point estimate that is off by 239 Elo on average is only useful if it comes with an honest *range*.
We train three LightGBM quantile models (0.05 / 0.50 / 0.95) for a raw 90% interval — but quantile
regression is not guaranteed to be calibrated, and here it **under-covers** (empirical 87.4% for a
nominal 90%). **Split conformal prediction** (Romano, Patterson & Candès, *NeurIPS* 2019) fixes this
with a finite-sample guarantee:

- Hold out a **calibration** split, never seen by any model fit (`model.calib_size`).
- On it, compute each point's nonconformity score `E = max(lower − y, y − upper)`.
- The correction is the `k`-th smallest score, `k = ⌈(n+1)(1−α)⌉` — widen (or narrow) the interval by
  it. This *guarantees* ≥ 90% coverage on exchangeable test data, regardless of the model's quality.

In [ ]:
# The finite-sample conformal quantile, using the actual src function on an illustrative calib set.
from src.model import _conformal_quantile, conformal_correction

rng = np.random.RandomState(0)
y_cal = rng.normal(1600, 300, 400)
med = y_cal + rng.normal(0, 150, 400)                 # a shrunk-ish point prediction
cal = {"lower": med - 250, "median": med, "upper": med + 250}   # a too-narrow raw 90% interval
alpha = 0.10
E = np.maximum(cal["lower"] - y_cal, y_cal - cal["upper"])       # nonconformity scores
k = math.ceil((len(E) + 1) * (1 - alpha))
q = _conformal_quantile(E, 1 - alpha)
lo, hi = conformal_correction(cal, y_cal, alpha)
print(f"illustrative calibration: n={len(E)}, k=ceil((n+1)*0.9)={k}")
print(f"conformal correction = {k}-th smallest nonconformity score = {q:+.1f} Elo  -> widen both sides")
print(f"conformal_correction() returns (lo, hi) = ({lo:+.1f}, {hi:+.1f})")

# The REAL effect on the 300k test set (from the artifact):
cov = pd.DataFrame({
    "coverage %": [A["coverage_raw"] * 100, A["coverage_cqr_plain"] * 100, A["coverage_cqr_mondrian"] * 100],
    "mean width (Elo)": [A["mean_interval_width_raw"], A["mean_interval_width_plain"], A["mean_interval_width_mondrian"]],
}, index=["raw quantile", "+ split CQR (plain)", "+ Mondrian CQR"]).round(1)
display(cov)

In [ ]:
# Coverage before/after conformalization, with the 90% target.
labels = ["raw\nquantile", "split CQR\n(plain)", "Mondrian\nCQR"]
vals = [A["coverage_raw"] * 100, A["coverage_cqr_plain"] * 100, A["coverage_cqr_mondrian"] * 100]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
bars = ax.bar(labels, vals, color=["#c0504d", "#2c6fbb", "#3aa66f"])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%", ha="center", va="bottom", fontweight="bold")
ax.axhline(90, color="k", ls="--", lw=1.2); ax.text(2.4, 90.2, "90% target", fontsize=9, ha="right")
ax.set(ylabel="Empirical coverage (%)", title="CQR restores the coverage guarantee", ylim=(80, 95))
show(save_fig(fig, "improved_cqr_coverage", FIG))
print(f"plain correction lo=hi = {A['conformal_correction'][0]:.1f} Elo   (interval width {A['mean_interval_width_raw']:.0f} -> {A['mean_interval_width_plain']:.0f})")

**Reading it.** The raw quantile interval covers only 87.4% — it lies about its own confidence. Split
CQR adds a modest ±27 Elo to each side and restores **89.8%** (target 90) with a *guaranteed*
finite-sample floor. **Mondrian CQR** (a separate correction per *predicted*-median band) reaches
89.9% at essentially the same width. The interval is now honest: when the model says "1600 ± 500," it
means it, marginally.

## 4. Where the guarantee frays: per-band coverage & the confusion matrix

Marginal coverage is a promise *on average over all players*. The band-conditional view exposes where
the interval is too tight — at the rating extremes — which is the seam notebook 05 pulls on.

In [ ]:
# Band confusion matrix (from the live pipeline) + per-band coverage, plain vs Mondrian.
show(FIGROOT / "band_confusion.png")

pb = pd.DataFrame(A["coverage_by_band_plain"])[["band", "coverage"]].rename(columns={"coverage": "plain"})
mb = pd.DataFrame(A["coverage_by_band_mondrian"])[["band", "coverage"]].rename(columns={"coverage": "mondrian"})
cov_band = pb.merge(mb, on="band")
x = np.arange(len(cov_band))
fig, ax = plt.subplots(figsize=(9, 4.3))
ax.bar(x - 0.2, cov_band["plain"] * 100, width=0.4, label="plain CQR", color="#2c6fbb")
ax.bar(x + 0.2, cov_band["mondrian"] * 100, width=0.4, label="Mondrian CQR", color="#e07b39")
ax.axhline(90, color="k", ls="--", lw=1.2)
ax.set_xticks(x); ax.set_xticklabels(cov_band["band"], rotation=20)
ax.set(ylabel="Coverage (%)", title="Per-band coverage: the extremes under-cover", ylim=(60, 105))
ax.legend()
show(save_fig(fig, "improved_coverage_by_band", FIG))

**Reading it.** Coverage is near-perfect in the populous middle bands (95–98%) but sags to **~78% at
0–1200 and ~81% at 2000+** — and Mondrian CQR barely changes this (78/81 either way). The confusion
matrix tells the same story: predictions concentrate on the diagonal in the centre and smear toward
the middle at the extremes (36.4% exact-band, 75.9% adjacent-band accuracy). *Why* Mondrian can't fix
the tail — even though it is the theoretically right tool — is closed as a formal negative result in
notebook 05 (the predicted extreme bands are nearly empty, so no test-time-legal recalibration can
reach them).

## 5. The headline: the aggregation curve

The single-game noise floor is real, but it is not the end of the story. Averaging a player's per-game
predictions collapses the noise. This is the chart the whole project builds to.

In [ ]:
curve = pd.DataFrame(A["aggregation_curve"])
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(curve["k"], curve["mae"], "-o", color="#2c6fbb", ms=6, lw=2)
for _, row in curve.iterrows():
    ax.annotate(f"{row['mae']:.0f}\n(n={int(row['n_players']):,})", (row["k"], row["mae"]),
                textcoords="offset points", xytext=(0, 10), ha="center", fontsize=8)
ax.set(xlabel="Games per player (K), predictions averaged", ylabel="MAE (Elo)",
       title="Aggregation curve: single-game noise collapses with a few games")
ax.set_ylim(180, 250)
show(save_fig(fig, "improved_aggregation_curve", FIG))
display(curve.rename(columns={"mae": "MAE", "n_players": "players with >=K games"}).round(1))

**Reading it.** One game gives **MAE ~235**; averaging just **five** games drops it to **~200**, after
which the curve flattens and even ticks up — but read the `n_players` column: each higher-K point is a
*different, shrinking* cohort (50,357 players at K=1 down to 234 at K=20), so the K=10–20 wobble is
cohort noise, not a real reversal. This is exactly the confound the **player-cohort** sample was built
to remove (notebook 01, §3). The actionable takeaway stands: **a single blitz game pins a rating to
±~235 Elo; a handful of games tightens it to ~200 — and the interval around each estimate is
*guaranteed*, not hoped for.**

## Takeaways

- **Engine + clock features are the win:** 293 → 239 MAE, with move-quality (the engine group)
  worth +33.6 on its own — more than every other group combined.
- **The interval is honest, by construction.** Split CQR converts an under-covering 87.4% raw
  interval into a *guaranteed* ~90% marginal one; Mondrian matches it at equal width.
- **The guarantee is marginal, not conditional.** Coverage sags at the rating extremes (78% / 81%) —
  the interval-space signature of the single-game noise floor.
- **Aggregation is the real lever:** ±235 Elo from one game, ~200 from five.

**Next:** notebook 05 asks the uncomfortable question — *is that tail error a bug we can fix, or a
noise floor we cannot?* — and closes three independent hypotheses to answer it.